### Imports & Setup

In [13]:
import pandas as pd
import json

from pubmed import PubMedRetriever
from vector_store import VectorStore
from summarizer import LLMSummarizer
from config import TOP_K_RESULTS 

### Retrieve PMIDs & Fetch Articles

In [15]:
search_term = "gene therapy cancer"  
max_results = 10

pmids = PubMedRetriever.search_pubmed_articles(search_term, max_results=max_results)
print(f"Retrieved {len(pmids)} PMIDs")

articles = PubMedRetriever.fetch_pubmed_abstracts(pmids)
print(f"Fetched {len(articles)} article metadata/abstracts")

Retrieved 10 PMIDs
[{'pmid': '41159923', 'title': 'Precision design of ferritin nanocages: tumor-targeted delivery, multi-mechanism synergistic therapy and clinical translational challenges.', 'abstract': {'OBJECTIVE': 'To elucidate the application potential of ferritin nanocarriers in targeted cancer therapy, particularly for central nervous system (CNS) tumors (e.g., glioblastoma), focusing on their structural advantages, drug-loading capacity, targeting strategies, and emerging therapeutic directions.', 'REVIEW SIGNIFICANCE': 'Ferritin, leveraging its natural biocompatibility, biodegradability, and unique cage-like structure, is a highly promising drug delivery platform. Its intrinsic ability to specifically bind the tumor-overexpressed transferrin receptor 1 (TfR1/CD71) and penetrate the blood-brain barrier (BBB) provides a unique solution for the precise delivery of chemotherapeutics, metal ions, gene therapy molecules, and novel therapeutics.', 'MAIN FINDINGS': 'Ferritin achieves

### Display in DataFrame 

In [17]:
# Convert list of dicts into DataFrame
df = pd.DataFrame(articles)

# Utility to shorten abstract text
def short_abstract(ab_dict, length=200):
    if isinstance(ab_dict, dict):
        combined = " | ".join([f"{k}: {v}" for k, v in ab_dict.items()])
    else:
        combined = ab_dict or ""
    return (combined[:length] + '…') if len(combined) > length else combined

df['short_abstract'] = df['abstract'].apply(lambda x: short_abstract(x, length=200))

display(df[['pmid', 'title', 'journal', 'publication_date', 'authors', 'short_abstract']])

,pmid,title,journal,publication_date,authors,short_abstract
0,41159923,Precision design of ferritin nanocages: tumor-...,Drug development and industrial pharmacy,2025,"Hailong Liu, Nan Yang, Yucui Wang, Tianhao Li,...",OBJECTIVE: To elucidate the application potent...
1,41159914,Use of Patient-Derived Organoids for Pleural M...,American journal of physiology. Lung cellular ...,2025,"Luca Volpini, Federica Monaco, Elisabetta Casa...",SUMMARY: Defining pre-clinical models is of ut...
2,41159485,Targeting Mast Cell Activation and MIF-Mediate...,"Advanced science (Weinheim, Baden-Wurttemberg,...",2025,"Libo Wang, Guangcong Shen, Guanpeng Xie, Zekun...",SUMMARY: Neoadjuvant gemcitabine plus nab-pacl...
3,41159445,KLHL13 functional defects cause neurodevelopme...,Genetics in medicine : official journal of the...,2025,"Tehmeena Akhter, Zubair M Ahmed, Yaping Ji, Ax...",PURPOSE: Neurodevelopmental disorders (NDDs) a...
4,41159102,GPC2-CAR T cells have potent preclinical activ...,Molecular therapy. Oncology,2025,"Reona Okada, Mia Fanuzzi, Constanza Rodriguez,...",SUMMARY: Medulloblastoma (MB) is the most comm...
5,41159033,Next-generation T cell immunotherapy: overcomi...,Frontiers in immunology,2025,"Guangmei Li, Dengju Li, Xiaojian Zhu",SUMMARY: T-cells are a core component of tumor...
6,41159027,Chromosome 11q13 amplification as a decision-m...,Frontiers in immunology,2025,"Wen Jiang, Shengjin Dou, Lin Zhang, Minjun Don...",BACKGROUND: Anti-programmed cell death protein...
7,41158992,Operational burden and fragmented implementati...,HemaSphere,2025,"Nick van Sinderen, Julie Dolva, Caterina Riill...",SUMMARY: No Abstract
8,41158962,Bedside manufacturing of engineered stem cells...,Molecular therapy. Methods & clinical development,2025,"Carrie L Cummings, Sirkka B Stephan, Katelyn F...",SUMMARY: Hematopoietic stem cell (HSC) gene th...
9,41158839,Leveraging the htsFLT01/MiRGD Complex to Enhan...,Iranian journal of medical sciences,2025,"Mohadeseh Khoshandam, Zahra-Soheila Soheili, S...",BACKGROUND: Gene therapy introduces therapeuti...


### Prepare Documents & Ingest into Vector Store

In [19]:
# Cell: Prepare Documents & Ingest into Vector Store (Updated)

# 1. Prepare documents, IDs & metadata
docs = [
    f"{a['title']} - { ' '.join(a['abstract'].values()) if isinstance(a['abstract'], dict) else a['abstract'] }"
    for a in articles
]
ids = [ a['pmid'] for a in articles ]
metadatas = [
    {
        "pmid":            a['pmid'],
        "journal":         a['journal'],
        "publication_date":a['publication_date'],
        "authors":         a['authors']
    }
    for a in articles
]

# 2. Instantiate vector store with updated client config
from vector_store import VectorStore

vs = VectorStore(persist_directory="chroma_db", model_name="sentence-transformers/all-MiniLM-L6-v2")

# 3. Ingest documents
vs.ingest(docs=docs, metadatas=metadatas, ids=ids)
print(f"Ingested {len(docs)} documents into vector store.")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Ingested 10 documents into vector store.


/opt/anaconda3/lib/python3.12/site-packages/torch/nn/modules/module.py:1786: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


### Query & Summarize Answer

In [21]:
question = "What are recent advances in gene therapy for cancer?"
results  = vs.query(question, k=TOP_K_RESULTS)

# Build docs list for summarizer
docs_for_summary = []
for idx, doc_text in enumerate(results["documents"][0]):
    doc_id  = results["ids"][0][idx]
    # you might parse title & abstract from doc_text split by -
    parts   = doc_text.split(" - ", 1)
    title   = parts[0]
    abstract= parts[1] if len(parts) > 1 else ""
    docs_for_summary.append({
        "pmid":      doc_id,
        "title":     title,
        "abstract":  abstract
    })

summarizer = LLMSummarizer()
answer    = summarizer.generate_answer(question, docs_for_summary)
print("🧠 Answer:")
print(answer)

/opt/anaconda3/lib/python3.12/site-packages/torch/nn/modules/module.py:1786: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


🧠 Answer:
Recent advances in gene therapy for cancer include the use of exosomes as a platform for delivering therapeutic payloads, such as small molecules, RNA, or CRISPR-Cas systems [PMID: 41155971]. This approach offers a highly specific and low-immunogenic alternative to viral vectors or synthetic nanoparticles. Additionally, gene therapy can introduce therapeutic genes into cancer cells to inhibit tumor growth or induce apoptosis, as seen in the use of the htsFLT01/MiRGD complex to enhance apoptosis and suppress angiogenesis in breast cancer cells [PMID: 41158839]. Furthermore, emerging approaches in T cell immunotherapy, such as CRISPR-Cas9-based CAR-T optimization, aim to enhance therapeutic efficacy by precise genome editing [PMID: 41159033]. These advances highlight the potential of gene therapy in cancer treatment, particularly in combination with other therapeutic strategies.
